# Phase 3 — SASRec Retrieval + FAISS

Build customer purchase sequences from cleaned transactions, train SASRec on the train window's sequences, extract item embeddings, build a FAISS inner-product index, and retrieve top-50 candidate items per customer.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch

from src.data.splits import build_windows
from src.models.retrieval.dataset import build_sessions, Vocab, pad_right
from src.models.retrieval.train import TrainConfig, train_sasrec, encode_sessions, save
from src.faiss.index import build_index, topk, l2_normalize, save_index

df = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'transactions_clean.parquet')
windows = build_windows(df, horizon_days=90, n_windows=3)
windows[0], windows[1], windows[2]

In [ ]:
# Train sequences = customers' purchase orders up to train.feature_end
train_sessions = build_sessions(df, windows[0].feature_end, min_len=5)
raw = [items for _, items in train_sessions]
print(f"Sessions: {len(raw):,}  |  median len: {np.median([len(s) for s in raw]):.0f}")

In [ ]:
config = TrainConfig(emb_dim=64, n_layers=2, n_heads=2, max_len=50, batch_size=256, epochs=15)
model, vocab = train_sasrec(raw, config)
print(f"Vocab size: {vocab.vocab_size:,}")

In [ ]:
# Build FAISS index over item embeddings (skip pad row)
model.eval()
with torch.no_grad():
    item_vecs = model.item_matrix().cpu().numpy()
item_vecs_n = l2_normalize(item_vecs)
index = build_index(item_vecs_n)
print(f"FAISS index built: {index.ntotal} items, dim={item_vecs_n.shape[1]}")

In [ ]:
# Retrieval for test-window customers: encode their pre-feature_end sequence,
# get user vector, FAISS top-50.
test_sessions = build_sessions(df, windows[2].feature_end, min_len=3)
enc = encode_sessions([s for _, s in test_sessions], vocab)

X = torch.tensor([pad_right(s, config.max_len, vocab.pad_id) for s in enc], dtype=torch.long)
with torch.no_grad():
    uvecs = model.user_vector(X).cpu().numpy()
uvecs = l2_normalize(uvecs)
scores, ids = topk(index, uvecs, k=50)
scores.shape, ids.shape

In [ ]:
# Persist artifacts
MODELS = PROJECT_ROOT / 'data' / 'features'
MODELS.mkdir(parents=True, exist_ok=True)
save(model, vocab, MODELS / 'sasrec')
save_index(index, MODELS / 'item_index.faiss')

# Save top-K candidates for the ranking stage
test_customers = [c for c, _ in test_sessions]
cand = pd.DataFrame({'customer_id': np.repeat(test_customers, 50),
                     'rank': np.tile(np.arange(50), len(test_customers)),
                     'item_internal_id': ids.flatten(),
                     'score': scores.flatten()})
# map internal id back to stock_code
id2item = vocab.id2item
cand['stock_code'] = cand['item_internal_id'].map(lambda i: id2item[i] if i < len(id2item) and id2item[i] is not None else None)
cand.to_parquet(MODELS / 'retrieval_candidates_test.parquet', index=False)
print(f"Saved top-50 retrieval for {len(test_customers):,} test customers.")